Scrivi jobs/01_bronze_ingest.py che:

- definisce lo schema di orders.jsonl con StructType
- legge il JSON con quello schema esplicito
- aggiunge una colonna ingestion_ts (timestamp corrente) e una colonna source_file
- scrive il risultato come Parquet in data/bronze/orders, partizionato per una colonna sensata (pensaci: quale colonna useresti per partizionare dati che cresceranno ogni giorno, e perché?)

In [3]:
from pyspark.sql import SparkSession

In [4]:
spark = (SparkSession.builder
            .appName("shopflow-hello")
            .master("local[*]")
            .getOrCreate())

26/09/24 09:55:34 WARN Utils: Your hostname, codespaces-e51f72 resolves to a loopback address: 127.0.0.1; using 10.0.12.228 instead (on interface eth0)
26/09/24 09:55:34 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/24 09:55:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
from pathlib import Path

PROJECT_ROOT = Path("/workspaces/codespaces-blank")

In [6]:
raw = spark.read.text (str(PROJECT_ROOT / "data/raw/orders.jsonl"))
raw.show(5, truncate=False)

+----------------------------------------------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                                                           |
+----------------------------------------------------------------------------------------------------------------------------------------------------------------+
|{"order_id": 1, "customer_id": 1625, "product_id": 39, "quantity": 5, "status": "completed", "channel": "marketplace", "order_ts": "2026-09-02T00:58:31.400380"}|
|{"order_id": 2, "customer_id": 835, "product_id": 155, "quantity": 5, "status": "completed", "channel": "marketplace", "order_ts": "2026-07-18T04:45:01.400380"}|
|{"order_id": 3, "customer_id": 337, "product_id": 178, "quantity": 5, "status": "completed", "channel": "app", "order_ts": "2026-09-08T14:32:34.400380"}        |
|{"order_id": 4, "cust

In [7]:
orders = spark.read.json(str(PROJECT_ROOT / "data/raw/orders.jsonl"))

In [8]:
orders.printSchema()

root
 |-- channel: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- order_id: long (nullable = true)
 |-- order_ts: string (nullable = true)
 |-- product_id: long (nullable = true)
 |-- quantity: long (nullable = true)
 |-- status: string (nullable = true)



In [9]:
orders.show(5, truncate=False)

+-----------+-----------+--------+--------------------------+----------+--------+---------+
|channel    |customer_id|order_id|order_ts                  |product_id|quantity|status   |
+-----------+-----------+--------+--------------------------+----------+--------+---------+
|marketplace|1625       |1       |2026-09-02T00:58:31.400380|39        |5       |completed|
|marketplace|835        |2       |2026-07-18T04:45:01.400380|155       |5       |completed|
|app        |337        |3       |2026-09-08T14:32:34.400380|178       |5       |completed|
|app        |4145       |4       |2026-07-01T07:33:04.400380|175       |1       |completed|
|app        |3552       |5       |2026-07-04T20:47:37.400380|186       |4       |completed|
+-----------+-----------+--------+--------------------------+----------+--------+---------+
only showing top 5 rows



definiamo lo schema di orders.jsonl con StructType

In [10]:
from pyspark.sql.types import (
    StructType,
    StructField,
    
    StringType, 
    IntegerType, 
    DoubleType, 
    TimestampType
)

In [11]:
orders_schema = StructType([
    StructField("channel", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("order_id", StringType(), False),
    StructField("order_ts", TimestampType(), False),
    StructField("product_id", StringType(), False),
    StructField("quantity", IntegerType(), True),
    StructField("status", StringType(), True)
])

leggiamo il JSON con quello schema esplicito

In [14]:
orders = spark.read.schema(orders_schema).json(str(PROJECT_ROOT / "data/raw/orders.jsonl"))
orders.printSchema()
orders.show(5, truncate=False)

root
 |-- channel: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_ts: timestamp (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- status: string (nullable = true)

+-----------+-----------+--------+-------------------------+----------+--------+---------+
|channel    |customer_id|order_id|order_ts                 |product_id|quantity|status   |
+-----------+-----------+--------+-------------------------+----------+--------+---------+
|marketplace|1625       |1       |2026-09-02 00:58:31.40038|39        |5       |completed|
|marketplace|835        |2       |2026-07-18 04:45:01.40038|155       |5       |completed|
|app        |337        |3       |2026-09-08 14:32:34.40038|178       |5       |completed|
|app        |4145       |4       |2026-07-01 07:33:04.40038|175       |1       |completed|
|app        |3552       |5       |2026-07-04 20:47:37.40038|186       |

aggiungiamo una colonna ingestion_ts (timestamp corrente) e una colonna source_file
